# Wikidata properties extraction - Pliny's Wall of Fame


This code extracts the wikidata property values attached to people cited in books 2-6 of the _Naturalis Historia_ by Pliny the Elder. 

The code is made available together with the paper _Pliny’s “Wall of Fame”: A Digital Semantic Approach to Citations and Character Networks in Books 2-6 of the Naturalis Historia_, published in DH Benelux Journal.

The wikidata property values are extracted using the wikidata Python library (https://pypi.org/project/Wikidata/).
The properties extracted are the following:

- P31 - Instance of
- P27 - Country of citizenship
- P1412 - Languages spoken, written or signed
- P106 - Occupation
- P21 - Sex or gender

Adjust the parameters of the code to your needs.


## Imports

In [1]:
# import requests
import pandas as pd
import time
import os
from wikidata.client import Client

## Load data

We upload the original file, already including the Wikidata dates as extracted from de Graaf's dataset.

In [12]:
df = pd.read_csv('nh_original_dates.csv')

C:\Users\u0172188\AppData\Local\Temp\ipykernel_7712\1787617594.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('nh_original_dates.csv')


## Wikidata Extraction

This code extracts the selected Wikidata properties values. It saves a checkpoint every 100 entities: this number can be adjusted.
The total extraction requires around 30 minutes

In [14]:
#create a list of wikidata id
wikidata_ids = df["wikidata_qid"].dropna().unique().tolist()

In [16]:
#the extraction requires around 30 minutes
#----------------------------------------------------------
#create a Client from the wikidata python library
client = Client()

#identify the user-agent
client.opener.addheaders = [('User-Agent', 'PlinysWallofFame (https://www.wikidata.org/wiki/User:[insert])' )]

#Configuration
DELAY_BETWEEN_REQUESTS = 0.5  # seconds (adjust if needed)
CHECKPOINT_INTERVAL = 100  # Save progress every N entities
CHECKPOINT_FILE ="wikidata_extraction_checkpoint.csv"

#---------DEFINE FUNCTIONS------------------------------------

def get_label(client, qid):
    """Helper function to get English label for a Wikidata entity"""
    entity = client.get(qid, load=True) #get the entity data (entity.label, entity.claims ...)
    label = getattr(entity, 'label', None) #get the label among the various attributes (label = entity.label)
    if hasattr(label, 'get'):
        label = label.get('en') or next(iter(label.values()), None) #if english not available, take the first available
    return str(label) if label else None



def extract_property_values(claims, property_id, client):
    """Extract values and labels for a given property""" 
    #propriety claims = list of values associated with the property indicated by property id
    property_claims = claims.get(property_id, []) #claims = dictionary with all the properties wikidata
    ids, labels = [], []
    
    for claim in property_claims:
        mainsnak = claim.get('mainsnak', {})
        datavalue = mainsnak.get('datavalue', {})
        value = datavalue.get('value', {})
        
      
        if isinstance(value, dict):
            qid_target = value.get('id') #take the id of the value
            if qid_target:
                ids.append(qid_target)
                label = get_label(client, qid_target)
                labels.append(label)

    
    return ids, labels


def create_empty_result(qid):
    """Create an empty result dictionary"""
    return {
        "wikidata_qid": qid,
        "instance_of_id": None,
        "instance_of_label": None,
        "is_human": None,
        "citizenship_id": None,
        "citizenship_label": None,
        "language_id": None,
        "language_label": None,
        "occupation_id": None,
        "occupation_label": None,
        "sex_or_gender_id": None,
        "sex_or_gender_label": None
    }


def process_entity(client, qid):
    """Process a single Wikidata entity"""
  
        # Validate Wikidata ID format
    if not isinstance(qid, str) or not qid.startswith("Q"):
        return create_empty_result(qid)
    
    entity = client.get(qid, load=True) #load the page of the entity
    raw_data = entity.data
    claims = raw_data.get('claims', {})
    
    # P31 - Instance of
    instance_ids, instance_labels = extract_property_values(claims, 'P31', client)
    #define humans
    is_human = any(q in instance_ids for q in ["Q5", "Q21070568"])
    
    # P27 - Country of citizenship
    citizenship_ids, citizenship_labels = extract_property_values(claims, 'P27', client)
    
    # P1412 - Languages used
    language_ids, language_labels = extract_property_values(claims, 'P1412', client)
    
    # P106 - Occupation
    occupation_ids, occupation_labels = extract_property_values(claims, 'P106', client)
    
    # P21 - Sex or gender
    sex_or_gender_ids, sex_or_gender_labels = extract_property_values(claims, 'P21', client)
    
    return {
        "wikidata_qid": qid,
        "instance_of_id": "; ".join(instance_ids) if instance_ids else None,
        "instance_of_label": "; ".join(filter(None, instance_labels)) if instance_labels else None,
        "is_human": is_human,
        "citizenship_id": "; ".join(citizenship_ids) if citizenship_ids else None,
        "citizenship_label": "; ".join(filter(None, citizenship_labels)) if citizenship_labels else None,
        "language_id": "; ".join(language_ids) if language_ids else None,
        "language_label": "; ".join(filter(None, language_labels)) if language_labels else None,
        "occupation_id": "; ".join(occupation_ids) if occupation_ids else None,
        "occupation_label": "; ".join(filter(None, occupation_labels)) if occupation_labels else None,
        "sex_or_gender_id": "; ".join(sex_or_gender_ids) if sex_or_gender_ids else None,
        "sex_or_gender_label": "; ".join(filter(None, sex_or_gender_labels)) if sex_or_gender_labels else None
    }


#---------------------------------------------- Main processing loop------------------------------------------------------
results = []
total_qids = len(wikidata_ids)
start_index = 0

# Check for existing checkpoint
if os.path.exists(CHECKPOINT_FILE):
    print(f"Found checkpoint file: {CHECKPOINT_FILE}")
    response = input("Resume from checkpoint? (y/n): ").strip().lower()
    if response == 'y':
        checkpoint_df = pd.read_csv(CHECKPOINT_FILE)
        results = checkpoint_df.to_dict('records')
        start_index = len(results)
        print(f"Resuming from entity {start_index + 1}/{total_qids}")

print(f"\nStarting processing of {total_qids} Wikidata IDs...")
print(f"Delay between requests: {DELAY_BETWEEN_REQUESTS}s")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL} entities\n")

for idx in range(start_index, total_qids):
    qid = wikidata_ids[idx]
    
    # Progress indicator
    if (idx + 1) % 10 == 0:
        elapsed_pct = ((idx + 1) / total_qids) * 100
        print(f"Progress: {idx + 1}/{total_qids} ({elapsed_pct:.1f}%) - Current: {qid}")
    
    
#------------------- Process entity
    result = process_entity(client, qid)
    results.append(result)
    
    # Save checkpoint periodically
    if (idx + 1) % CHECKPOINT_INTERVAL == 0:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_csv(CHECKPOINT_FILE, index=False)
        print(f"  ✓ Checkpoint saved at {idx + 1} entities")
    
    # Rate limiting delay
    if idx < total_qids - 1:  # Don't delay after last item
        time.sleep(DELAY_BETWEEN_REQUESTS)

# Create DataFrame and merge with original data
df_extended = pd.DataFrame(results)
all_instances_extended = df.merge(df_extended, on="wikidata_qid", how="left")
all_instances_extended.to_csv("nh_extended_withdates.csv", index=False)

# Clean up checkpoint file
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
    print(f"\n✓ Checkpoint file removed")

print(f"\n=== SUMMARY ===")
print(f"Total QIDs to process: {total_qids}")
print(f"Successfully processed: {len(results)}")
print(f"Results saved to: nh_extended_withdates.csv")


Starting processing of 367 Wikidata IDs...
Delay between requests: 0.5s
Checkpoint interval: 100 entities

Progress: 10/367 (2.7%) - Current: Q605139
Progress: 20/367 (5.4%) - Current: Q4654
Progress: 30/367 (8.2%) - Current: Q673402
Progress: 40/367 (10.9%) - Current: Q1124866
Progress: 50/367 (13.6%) - Current: Q1413
Progress: 60/367 (16.3%) - Current: Q51673
Progress: 70/367 (19.1%) - Current: Q2933380
Progress: 80/367 (21.8%) - Current: Q897523
Progress: 90/367 (24.5%) - Current: Q200859
Progress: 100/367 (27.2%) - Current: Q355778
  ✓ Checkpoint saved at 100 entities
Progress: 110/367 (30.0%) - Current: Q673550
Progress: 120/367 (32.7%) - Current: Q88976791
Progress: 130/367 (35.4%) - Current: Q1407
Progress: 140/367 (38.1%) - Current: Q1199937
Progress: 150/367 (40.9%) - Current: Q4318
Progress: 160/367 (43.6%) - Current: Q510193
Progress: 170/367 (46.3%) - Current: Q1473861
Progress: 180/367 (49.0%) - Current: Q179844
Progress: 190/367 (51.8%) - Current: Q176758
Progress: 200/3